# Useful notebooks:

- Preprocessing : https://www.kaggle.com/code/motono0223/js24-preprocessing-create-lags
- Training (XGB) : https://www.kaggle.com/code/motono0223/js24-train-gbdt-model-with-lags-singlemodel
  - trained XGB model : https://www.kaggle.com/datasets/motono0223/js24-trained-gbdt-model
- Training (NN): **this notebook** https://www.kaggle.com/code/voix97/jane-street-rmf-training-nn
  - trained NN model : https://www.kaggle.com/datasets/voix97/js-xs-nn-trained-model
- Inference of NN : https://www.kaggle.com/code/voix97/jane-street-rmf-nn-with-pytorch-lightning
- Inference of NN+XGB:  https://www.kaggle.com/code/voix97/jane-street-rmf-nn-xgb
- EDA(1) : https://www.kaggle.com/code/motono0223/eda-jane-street-real-time-market-data-forecasting
- EDA(2) : https://www.kaggle.com/code/motono0223/eda-v2-jane-street-real-time-market-forecasting

# Training Neural Networks (MLP) with PyTorch Lightning

In [19]:
import os
import pickle
import polars as pl
import numpy as np
import pandas as pd

# Load Data

In [20]:
input_path = './input_df'
TRAINING = True
feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)]
label_name = 'responder_6'
weight_name = 'weight'
train_name = os.path.join("./input_df/", "nn_input_df_with_lags.pickle")
valid_name = os.path.join("./input_df/", "nn_valid_df_with_lags.pickle")
if TRAINING and not os.path.exists(train_name):
    df = pl.scan_parquet(f"{input_path}/train.parquet").collect().to_pandas()
    valid = pl.scan_parquet(f"{input_path}/valid.parquet").collect().to_pandas()
    df = pd.concat([df, valid]).reset_index(drop=True)# A trick to boost LB from 0.0045->0.005
    with open(train_name, "wb") as w:
        pickle.dump(df, w)
    with open(valid_name, "wb") as w:
        pickle.dump(valid, w)
elif TRAINING:
    with open(train_name, "rb") as r:
        df = pickle.load(r)
    with open(valid_name, "rb") as r:
        valid = pickle.load(r)

In [21]:
# Show the shape (rows, columns) of train and valid data
print("Train data shape:", df.shape)
print("Valid data shape:", valid.shape)

# Print the column names
print("\nTrain columns:", df.columns.tolist())
print("Valid columns:", valid.columns.tolist())

# Show the first 5 rows of train and valid data
print("\nTrain data sample:")
display(df.head())

print("\nValid data sample:")
display(valid.head())

# # Optionally, show some basic info and statistics
# print("\nTrain data info:")
# df.info()

# print("\nBasic statistics of numeric columns in train data:")
# display(df.describe())


Train data shape: (22104280, 104)
Valid data shape: (1082224, 104)

Train columns: ['id', 'date_id', 'time_id', 'symbol_id', 'weight', 'feature_00', 'feature_01', 'feature_02', 'feature_03', 'feature_04', 'feature_05', 'feature_06', 'feature_07', 'feature_08', 'feature_09', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30', 'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35', 'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40', 'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45', 'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50', 'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55', 'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60', 'feature_61

,id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,...,label,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
0,25023058,1101,0,0,1.913215,0.458568,-0.052640,-0.044876,0.124223,2.819707,...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25023059,1101,0,1,4.360143,0.111398,0.500473,0.083353,0.395555,2.468412,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25023060,1101,0,2,1.259379,0.536855,0.309993,0.015098,0.318689,2.594687,...,-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25023061,1101,0,3,1.498643,-0.016845,-0.104391,-0.202445,0.404921,2.384175,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25023062,1101,0,4,1.013641,0.269194,-0.088913,-0.436358,-0.025382,2.440446,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Valid data sample:


,id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,...,label,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
0,46045114,1670,0,0,3.405333,0.310848,1.131483,0.745982,0.776556,-0.798888,...,0,-0.599179,-0.421537,-0.880075,0.227441,0.181244,-0.087156,0.213826,0.138748,0.242813
1,46045115,1670,0,1,2.619899,0.382185,0.458687,0.544353,0.092550,-0.911843,...,0,1.206133,0.458626,-0.101498,0.050520,0.018345,-0.553546,-1.088641,-0.553292,-1.762422
2,46045116,1670,0,2,2.213198,-0.163449,0.740222,0.272134,-0.092383,-1.215421,...,0,-0.698454,-0.217275,-0.452308,0.005591,0.004208,-0.046340,-0.468561,-0.158282,-0.714670
3,46045117,1670,0,3,1.603719,0.362074,0.242651,0.463201,0.282871,-1.410926,...,3,2.172535,1.317854,-0.800447,0.191972,0.158641,-0.213527,-0.072624,-0.007186,-0.156377
4,46045118,1670,0,5,2.820571,-0.015899,1.502831,0.238775,0.330908,-0.765390,...,-1,1.545638,1.421872,0.804765,0.064878,0.096504,0.067876,-0.366895,-0.114120,-0.340877


In [22]:
# 1) Define helper function
def add_moving_averages(df, columns_to_transform, date_col='date_id', window=5, min_periods=1):
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        df[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    return df

# 3) Select which features to compute MAs for
features_for_ma = [f"feature_{i:02d}" for i in range(79)]
df = add_moving_averages(df, features_for_ma)
valid = add_moving_averages(valid, features_for_ma)

# 4) Decide if you want these new MA features for your model
use_ma_features = True

# Original feature names
feature_names = [f"feature_{i:02d}" for i in range(79)] + \
                [f"responder_{idx}_lag_1" for idx in range(9)]

if use_ma_features:
    ma_feature_names = [f"{col}_ma5" for col in features_for_ma]
    feature_names += ma_feature_names

# 5) Set label & weight
label_name = 'responder_6'
weight_name = 'weight'

In [23]:
X_train = df[ feature_names ]
y_train = df[ label_name ]
w_train = df[ "weight" ]
X_valid = valid[ feature_names ]
y_valid = valid[ label_name ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

((22104280, 167),
 (22104280,),
 (22104280,),
 (1082224, 167),
 (1082224,),
 (1082224,))

# Training Configurations

In [24]:
import os
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import (LightningDataModule, LightningModule, Trainer)
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader


class custom_args():
    def __init__(self):
        self.usegpu = True
        self.gpuid = 1
        self.seed = 42
        self.model = 'nn'
        self.use_wandb = False
        self.project = 'js-xs-nn-with-lags'
        self.dname = "./input_df/"
        self.loader_workers = 4
        self.bs = 8192
        self.lr = 1e-3
        self.weight_decay = 5e-4
        self.dropouts = [0.1, 0.1]
        self.n_hidden = [512, 512, 256]
        self.patience = 25
        self.max_epochs = 30
        self.N_fold = 5


my_args = custom_args()

# PyTorch Data Module Definition

In [25]:
class CustomDataset(Dataset):
    def __init__(self, df, accelerator):
        self.features = torch.FloatTensor(df[feature_names].values).to(accelerator)
        self.labels = torch.FloatTensor(df[label_name].values).to(accelerator)
        self.weights = torch.FloatTensor(df[weight_name].values).to(accelerator)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        w = self.weights[idx]
        return x, y, w


class DataModule(LightningDataModule):
    def __init__(self, train_df, batch_size, valid_df=None, accelerator='cpu'):
        super().__init__()
        self.df = train_df
        self.batch_size = batch_size
        self.dates = self.df['date_id'].unique()
        self.accelerator = accelerator
        self.train_dataset = None
        self.valid_df = None
        if valid_df is not None:
            self.valid_df = valid_df
        self.val_dataset = None

    def setup(self, fold=0, N_fold=5, stage=None):
        # Split dataset
        selected_dates = [date for ii, date in enumerate(self.dates) if ii % N_fold != fold]
        df_train = self.df.loc[self.df['date_id'].isin(selected_dates)]
        self.train_dataset = CustomDataset(df_train, self.accelerator)
        if self.valid_df is not None:
            df_valid = self.valid_df
            self.val_dataset = CustomDataset(df_valid, self.accelerator)

    def train_dataloader(self, n_workers=0):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=n_workers)

    def val_dataloader(self, n_workers=0):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=n_workers)
    


# NN Model Definition

In [26]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))
            if i > 0:
                layers.append(nn.SiLU())
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            layers.append(nn.Linear(in_dim, hidden_dim))
            # layers.append(nn.ReLU())
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, 1)) 
        layers.append(nn.Tanh())
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)  

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w  #
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation WRMSE at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        if self.trainer.sanity_checking:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
            # r2_val
            val_r_square = r2_val(y, prob, weights)
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

# Create PyTorch Data Module

In [27]:

args = my_args

# checking device
device = torch.device(f'cuda:{args.gpuid}' if torch.cuda.is_available() and args.usegpu else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() and args.usegpu else 'cpu'
loader_device = 'cpu'


# Initialize Data Module

df[feature_names] = df[feature_names].fillna(method = 'ffill').fillna(0)
valid[feature_names] = valid[feature_names].fillna(method = 'ffill').fillna(0)
data_module = DataModule(df, batch_size=args.bs, valid_df=valid, accelerator=loader_device)

# Create Model and Training

In [ ]:
import gc
del df
gc.collect()
pl.seed_everything(args.seed)
for fold in range(args.N_fold):
    data_module.setup(fold, args.N_fold)
    # Obtain input dimension
    input_dim = data_module.train_dataset.features.shape[1]
    # Initialize Model
    model = NN(
        input_dim=input_dim,
        hidden_dims=args.n_hidden,
        dropouts=args.dropouts,
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    # Initialize Logger
    if args.use_wandb:
        wandb_run = wandb.init(project=args.project, config=vars(args), reinit=True)
        logger = WandbLogger(experiment=wandb_run)
    else:
        logger = None
    # Initialize Callbacks
    early_stopping = EarlyStopping('val_loss', patience=args.patience, mode='min', verbose=False)
    checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1, verbose=False, filename=f"./models/nn_{fold}.model") 
    timer = Timer()
    # Initialize Trainer
    trainer = Trainer(
        max_epochs=args.max_epochs,
        accelerator=accelerator,
        devices=[args.gpuid] if args.usegpu else None,
        logger=logger,
        callbacks=[early_stopping, checkpoint_callback, timer],
        enable_progress_bar=True
    )
    # Start Training
    trainer.fit(model, data_module.train_dataloader(args.loader_workers), data_module.val_dataloader(args.loader_workers))
    # You can find trained best model in your local path
    print(f'Fold-{fold} Training completed in {timer.time_elapsed("train"):.2f}s')


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jw3621/Js_train/js-venv/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 482 K  | train
---------------------------------------------
482 K     Trainable params
0         Non-trainable params
482 K     Total params
1.931     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 29: 100%|██████████| 2159/2159 [02:48<00:00, 12.81it/s, v_num=4, val_r_square=0.0108]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: {'val_loss': '1.06156', 'val_r_square': '0.01079', 'train_loss': '1.47607'}
Epoch 29: 100%|██████████| 2159/2159 [02:48<00:00, 12.81it/s, v_num=4, val_r_square=0.0108]
Fold-0 Training completed in 4732.23s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jw3621/Js_train/js-venv/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 482 K  | train
---------------------------------------------
482 K     Trainable params
0         Non-trainable params
482 K     Total params
1.931     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 29: 100%|██████████| 2158/2158 [02:29<00:00, 14.42it/s, v_num=5, val_r_square=0.00843]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: {'val_loss': '1.06409', 'val_r_square': '0.00843', 'train_loss': '1.47500'}
Epoch 29: 100%|██████████| 2158/2158 [02:29<00:00, 14.42it/s, v_num=5, val_r_square=0.00843]
Fold-1 Training completed in 4655.41s


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jw3621/Js_train/js-venv/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 482 K  | train
---------------------------------------------
482 K     Trainable params
0         Non-trainable params
482 K     Total params
1.931     Total estimated model params size (MB)
13        Modules in train mode
0         Modules in eval mode


Epoch 8: 100%|██████████| 2158/2158 [02:31<00:00, 14.25it/s, v_num=6, val_r_square=0.00704]